# 多模態人工智慧應用

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 說明多模態 AI 如何整合文字、影像與感測資料。
2. 用輕量方式模擬 CLIP 的「圖文對齊」概念。
3. 比較早期融合與晚期融合在感測資料整合上的差異。
4. 理解語音助理中 ASR、NLU、回應生成的基本流程。
5. 以 Python 實作一個簡化版多模態決策流程。

本練習不使用大型模型，而是用 TF-IDF、數值特徵與規則式方法示範核心觀念。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立可重複執行的隨機種子。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

print('套件載入完成')
print('numpy version:', np.__version__)
print('pandas version:', pd.__version__)


## 核心概念說明

多模態 AI 是指能同時處理兩種以上資料模態的 AI 系統。常見模態包含：

- 文字：描述、問句、病歷、客服紀錄
- 影像：照片、醫學影像、監視器畫面
- 語音：語音內容、語調、背景聲音
- 感測資料：GPS、溫度、車速、心率、雷達訊號

多模態 AI 的關鍵不只是「把資料放在一起」，而是要讓不同模態能在語意上對齊。例如 CLIP 會把圖片與文字描述映射到同一個向量空間，使圖片「貓在草地上」能和文字描述「一隻貓在草地上」產生較高相似度。

在本 Notebook 中，我們會用 TF-IDF 模擬文字向量，用人工設計的影像標籤模擬圖片特徵，示範圖文對齊與融合決策。


In [ ]:
# ── 示範：用 TF-IDF 模擬圖文對齊 ──────────────────────
# 這段程式碼用文字特徵模擬 CLIP 的核心精神：把圖片描述與查詢文字轉成向量，再用餘弦相似度找出最相符的圖片。

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

images = pd.DataFrame({
    'image_id': ['img_001', 'img_002', 'img_003', 'img_004'],
    'visual_tags': [
        'cat grass animal outdoor',
        'red car road vehicle speed',
        'doctor xray hospital medical',
        'traffic camera city road congestion'
    ]
})

query = 'a vehicle on the road'

vectorizer = TfidfVectorizer()
all_text = images['visual_tags'].tolist() + [query]
vectors = vectorizer.fit_transform(all_text)

image_vectors = vectors[:-1]
query_vector = vectors[-1]

scores = cosine_similarity(query_vector, image_vectors).flatten()
images['match_score'] = scores
result = images.sort_values('match_score', ascending=False)

print('查詢文字:', query)
print(result[['image_id', 'visual_tags', 'match_score']])


## 多模態融合策略

多模態系統常見的融合方式包含：

1. 早期融合：先把不同模態的原始特徵合併，再交給模型判斷。例如把車流量、空氣品質、影像中的車輛數一起合併成一個特徵表。
2. 晚期融合：每個模態先各自產生判斷結果，再整合不同模型的分數。例如影像模型判斷塞車機率，感測器模型判斷塞車機率，最後加權平均。

早期融合容易實作，但需要處理尺度差異與資料同步。晚期融合較模組化，適合不同模態資料品質不穩定或更新頻率不同的情境。


In [ ]:
# ── 示範：智慧城市感測資料的早期融合 ────────────────────────
# 這段程式碼把影像偵測到的車輛數、道路平均速度與空氣品質指標合併，建立一個簡化的交通壅塞風險分數。

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

traffic = pd.DataFrame({
    'location': ['A路口', 'B路口', 'C路口', 'D路口'],
    'vehicle_count_from_image': [85, 30, 120, 55],
    'avg_speed_sensor': [18, 52, 10, 35],
    'air_quality_index': [90, 45, 110, 60]
})

features = traffic[['vehicle_count_from_image', 'avg_speed_sensor', 'air_quality_index']].copy()
features['inverse_speed'] = 1 / features['avg_speed_sensor']
features = features[['vehicle_count_from_image', 'inverse_speed', 'air_quality_index']]

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

weights = np.array([0.45, 0.35, 0.20])
traffic['congestion_risk_score'] = scaled @ weights
traffic['risk_level'] = pd.cut(
    traffic['congestion_risk_score'],
    bins=[-np.inf, -0.4, 0.6, np.inf],
    labels=['低', '中', '高']
)

print(traffic[['location', 'vehicle_count_from_image', 'avg_speed_sensor', 'air_quality_index', 'congestion_risk_score', 'risk_level']])


## 語音與文字模態的簡化應用流程

真實語音助理通常包含下列流程：

1. ASR：把語音轉成文字。
2. NLU：理解文字中的意圖。
3. 任務執行：查詢天氣、控制設備、搜尋資料或觸發服務。
4. 回應生成：產生文字或語音回應。

在本練習中，我們不處理真正的音訊，而是假設 ASR 已經把語音轉成文字，再用關鍵字規則模擬 NLU。這能幫助你理解語音模態整合到多模態系統時的基本設計。


## 🧪 自我測驗

請完成下方 TODO 填空，實作簡化版多模態客服判斷：同時參考使用者文字、語音壓力分數與臉部情緒分數，加權後判斷是否需要人工客服介入。


In [ ]:
# ── 🧪 自我測驗 ──────────────────────────────────
# 請完成下方 TODO 填空，實作一個簡化版多模態客服判斷：同時參考使用者文字、語氣分數與影像情緒分數，判斷是否需要人工客服介入。

import re
import numpy as np

user_text = '我等很久了，問題還是沒有解決，真的很生氣'
voice_stress_score = 0.82
face_negative_score = 0.76

negative_keywords = ['生氣', '等很久', '沒有解決', '投訴', '不滿']

# TODO 1: 計算文字中命中的負面關鍵字數量
keyword_hits = sum(1 for word in negative_keywords if word in user_text)

# TODO 2: 將關鍵字命中數轉成 0 到 1 之間的文字負面分數
text_negative_score = min(keyword_hits / 3, 1.0)

# TODO 3: 設定三種模態的權重，順序為文字、語音、臉部表情
weights = np.array([0.4, 0.35, 0.25])

scores = np.array([text_negative_score, voice_stress_score, face_negative_score])
final_risk = float(scores @ weights)
need_human_agent = final_risk >= 0.7

print('文字負面分數:', round(text_negative_score, 2))
print('語音壓力分數:', voice_stress_score)
print('臉部負面分數:', face_negative_score)
print('整合風險分數:', round(final_risk, 2))
print('是否轉接人工客服:', need_human_agent)

# Expected: 整合風險分數約大於 0.7
# Expected: 是否轉接人工客服: True
